In [ ]:
from langchain_community.vectorstores import FAISS
# from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [12]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [13]:
# Initialize OpenAI embeddings

embedding_model = OpenAIEmbeddings()

# Create FAISS vector store

vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [14]:
# Create retrievers

similarity_retriever = vectorstore.as_retriever(search_type="similarity",search_kwargs={"k":5} )

In [42]:
# multiquery_retriever = MultiQueryRetriever.from_llm(
#     retriever=vectorstore.as_retriever( search_kwargs={"k": 5}),
#     llm=ChatOpenAI(model="gpt-3.5-turbo"),
# )
# multiquery_retriever = MultiQueryRetriever.from_llm(
#     retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 4}),
#     llm=ChatOpenAI(model="gpt-4o-mini"),
# )
multiquery_retriever = vectorstore.as_retriever(
    search_type="mmr", # <- This enables MMR
    search_kwargs={"k":5, "lambda_mult":1} # k = top results, lambda mult = relevance-diversity balance
)

In [43]:
# Query

query = 'How to improve energy levels and maintain balance?'

In [44]:
# Retrieve results

similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [45]:
for i, doc in enumerate(similarity_results):
    print(f"\n-- Similarity Result {i+1} --")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n-- Multiquery Result {i+1} --")
    print(doc.page_content)


-- Similarity Result 1 --
Drinking sufficient water throughout the day helps maintain metabolism and energy.

-- Similarity Result 2 --
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

-- Similarity Result 3 --
Regular walking boosts heart health and can reduce symptoms of depression.

-- Similarity Result 4 --
Deep sleep is crucial for cellular repair and emotional regulation.

-- Similarity Result 5 --
The solar energy system in modern homes helps balance electricity demand.
******************************************************************************************************************************************************

-- Multiquery Result 1 --
Drinking sufficient water throughout the day helps maintain metabolism and energy.

-- Multiquery Result 2 --
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

-- Multiquery Result 3 --
Regular walking boosts heart health and can reduce symptoms of depression.

-- Multiquery 